# LENR Alternative Physics — ML Training Pipeline

**3 физических режима**: Maxwell (стандарт), Coulomb Original (1785), Cherepanov (фотонная масса)

**ML модели**:
1. XGBoost Classifier — бинарная классификация: произойдёт ли реакция?
2. DNN Regressor — предсказание избыточного тепла (W) с physics-informed loss
3. Anomaly Detector — фильтрация выбросов в данных

**Данные**: синтетические (из физического движка) + экспериментальные (Kasagi, McKubre, Fleischmann-Pons и др.)

In [ ]:
# Установка зависимостей (раскомментировать для Google Colab)
# !pip install xgboost shap torch scikit-learn numba matplotlib seaborn joblib

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Пути
ROOT = Path(os.getcwd()).parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
    sys.path.insert(0, str(ROOT.parent))  # для lenr_constants

print(f'Root: {ROOT}')
print(f'Python: {sys.version}')

# Настройки отображения
plt.style.use('dark_background')
sns.set_palette('husl')
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.4g}'.format)

## 1. Генерация данных

In [ ]:
from data_generator import LENRDataGenerator, get_feature_columns

gen = LENRDataGenerator(seed=42)

# Синтетические данные
N_SAMPLES = 10000
df_synthetic = gen.generate_parameter_sweep(n_samples=N_SAMPLES, noise_level=0.05)
print(f'Synthetic: {df_synthetic.shape}')
print(f'Reaction rate: {df_synthetic["reaction_occurred"].mean():.1%}')
print(f'Mean excess heat (>0): {df_synthetic[df_synthetic["excess_heat_W"] > 0]["excess_heat_W"].mean():.2f} W')

# Экспериментальные данные
df_experimental = gen.generate_experimental_dataframe()
print(f'\nExperimental: {df_experimental.shape}')
print(df_experimental[['lab', 'material', 'excess_W', 'COP']].to_string())

In [ ]:
# Обзор данных
feature_cols = get_feature_columns()
print(f'Features ({len(feature_cols)}): {feature_cols}')
print(f'\nСтатистика:')
df_synthetic[feature_cols].describe().T

## 2. Визуализация данных

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Распределение ключевых параметров', fontsize=14)

# Энергия пучка vs вероятность реакции
axes[0,0].scatter(df_synthetic['beam_energy_keV'], df_synthetic['reaction_probability'],
                  c=df_synthetic['reaction_occurred'], cmap='coolwarm', alpha=0.3, s=5)
axes[0,0].set_xlabel('E_cm (keV)')
axes[0,0].set_ylabel('P(reaction)')
axes[0,0].set_title('Энергия vs Вероятность')

# Загрузка D/Pd vs excess heat
axes[0,1].scatter(df_synthetic['deuterium_loading'], df_synthetic['excess_heat_W'],
                  c=df_synthetic['screening_energy_eV'], cmap='viridis', alpha=0.3, s=5)
axes[0,1].set_xlabel('D/Pd загрузка')
axes[0,1].set_ylabel('Excess Heat (W)')
axes[0,1].set_title('Загрузка vs Тепло')
axes[0,1].axvline(x=0.84, color='red', linestyle='--', alpha=0.7, label='McKubre threshold')
axes[0,1].legend(fontsize=8)

# Барьерные снижения: 3 режима
for i, mode in enumerate(['maxwell', 'coulomb', 'cherepanov']):
    col = f'barrier_reduction_{mode}'
    axes[0,2].hist(df_synthetic[col], bins=50, alpha=0.5, label=mode)
axes[0,2].set_xlabel('Barrier Reduction Ratio')
axes[0,2].set_title('Снижение барьера по режимам')
axes[0,2].legend(fontsize=8)

# log rate по режимам
for mode in ['maxwell', 'coulomb', 'cherepanov']:
    col = f'log_rate_{mode}'
    axes[1,0].hist(df_synthetic[col], bins=50, alpha=0.5, label=mode)
axes[1,0].set_xlabel('log10(rate)')
axes[1,0].set_title('Распределение log(rate)')
axes[1,0].legend(fontsize=8)

# Материалы vs реакция
mat_react = df_synthetic.groupby('material')['reaction_occurred'].mean().sort_values(ascending=False)
mat_react.plot.bar(ax=axes[1,1], color='#4fc3f7')
axes[1,1].set_title('Доля реакций по материалам')
axes[1,1].set_ylabel('Fraction')

# Корреляционная матрица
key_cols = ['screening_energy_eV', 'beam_energy_keV', 'temperature_K',
            'deuterium_loading', 'reaction_probability', 'excess_heat_W']
corr = df_synthetic[key_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1,2],
            xticklabels=[c[:12] for c in key_cols], yticklabels=[c[:12] for c in key_cols])
axes[1,2].set_title('Корреляции')

plt.tight_layout()
plt.show()

## 3. Детекция аномалий

In [ ]:
from models.anomaly_detector import LENRAnomalyDetector

detector = LENRAnomalyDetector(contamination=0.05)
anomaly_result = detector.fit_detect(df_synthetic, feature_cols)

print(f'Всего: {anomaly_result.n_total}')
print(f'Аномалий: {anomaly_result.n_anomalies} ({anomaly_result.anomaly_fraction:.1%})')
print(f'\nТоп фичи с аномалиями:')
for feat, count in list(anomaly_result.feature_anomaly_counts.items())[:10]:
    print(f'  {feat}: {count}')

# Физическая валидация
df_checked = detector.physics_sanity_check(df_synthetic)
print(f'\nФизически валидных: {df_checked["physics_valid"].sum()} / {len(df_checked)}')

In [ ]:
# Отфильтруем аномалии
df_clean = detector.filter_data(df_synthetic)
print(f'После фильтрации: {len(df_clean)} (было {len(df_synthetic)})')

## 4. XGBoost Classifier — Предсказание реакции

In [ ]:
from models.xgboost_classifier import LENRClassifier

classifier = LENRClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
)

clf_result = classifier.train(
    df_clean,
    feature_cols=feature_cols,
    target_col='reaction_occurred',
    test_size=0.2,
    do_cv=True,
)

print('=== XGBoost Classifier Results ===')
print(f'Accuracy:  {clf_result.accuracy:.4f}')
print(f'Precision: {clf_result.precision:.4f}')
print(f'Recall:    {clf_result.recall:.4f}')
print(f'F1:        {clf_result.f1:.4f}')
print(f'AUC-ROC:   {clf_result.auc_roc:.4f}')

if clf_result.cv_scores is not None:
    print(f'\nCV AUC-ROC: {clf_result.cv_scores.mean():.4f} +/- {clf_result.cv_scores.std():.4f}')

print(f'\nConfusion Matrix:\n{clf_result.confusion_matrix}')
print(f'\n{clf_result.classification_report}')

In [ ]:
# Feature importance
fig, ax = plt.subplots(figsize=(10, 8))
feat_imp = clf_result.feature_importance
y_pos = range(len(feat_imp))
ax.barh(y_pos, list(feat_imp.values()), color='#4fc3f7')
ax.set_yticks(y_pos)
ax.set_yticklabels(list(feat_imp.keys()), fontsize=9)
ax.set_xlabel('Importance (Gain)')
ax.set_title('XGBoost Feature Importance')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Analysis
try:
    import shap

    if clf_result.shap_values is not None:
        # Подготовим данные для SHAP plot
        from sklearn.model_selection import train_test_split
        X = df_clean[feature_cols].values.astype(np.float32)
        y = df_clean['reaction_occurred'].values
        X_scaled = classifier.scaler.transform(X)
        _, X_test, _, _ = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

        fig, axes = plt.subplots(1, 2, figsize=(18, 8))

        plt.sca(axes[0])
        shap.summary_plot(clf_result.shap_values, X_test,
                         feature_names=feature_cols, show=False, plot_size=None)
        axes[0].set_title('SHAP Summary Plot')

        plt.sca(axes[1])
        shap.summary_plot(clf_result.shap_values, X_test,
                         feature_names=feature_cols, plot_type='bar', show=False, plot_size=None)
        axes[1].set_title('SHAP Bar Plot')

        plt.tight_layout()
        plt.show()
    else:
        print('SHAP values not available')
except ImportError:
    print('shap not installed: pip install shap')

## 5. DNN Regressor — Предсказание избыточного тепла

In [ ]:
from models.dnn_regressor import LENRRegressor

regressor = LENRRegressor(
    hidden_dims=[128, 64, 32],
    lr=1e-3,
    batch_size=64,
    n_epochs=200,
    patience=20,
)

reg_result = regressor.train(
    df_clean,
    feature_cols=feature_cols,
    target_col='excess_heat_W',
    test_size=0.2,
    use_physics_loss=True,
)

print('=== DNN Regressor Results ===')
print(f'MSE:  {reg_result.mse:.4f}')
print(f'RMSE: {reg_result.rmse:.4f}')
print(f'MAE:  {reg_result.mae:.4f}')
print(f'R²:   {reg_result.r2:.4f}')
print(f'Epochs: {len(reg_result.train_losses)}')

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(reg_result.train_losses, label='Train', alpha=0.8)
axes[0].plot(reg_result.val_losses, label='Validation', alpha=0.8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Curves')
axes[0].legend()
axes[0].set_yscale('log')

# Feature importance (gradient-based)
feat_imp_reg = reg_result.feature_importance
y_pos = range(len(feat_imp_reg))
axes[1].barh(y_pos, list(feat_imp_reg.values()), color='#ff7043')
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(list(feat_imp_reg.keys()), fontsize=9)
axes[1].set_xlabel('Gradient Importance')
axes[1].set_title('DNN Feature Importance')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Pred vs Actual
from sklearn.model_selection import train_test_split

X = df_clean[feature_cols].values.astype(np.float32)
y = df_clean['excess_heat_W'].values
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

df_test_pred = pd.DataFrame(X_test, columns=feature_cols)
y_pred = regressor.predict(df_test_pred)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test, y_pred, alpha=0.3, s=10, color='#4fc3f7')
max_val = max(y_test.max(), y_pred.max())
ax.plot([0, max_val], [0, max_val], 'r--', alpha=0.7, label='Ideal')
ax.set_xlabel('Actual Excess Heat (W)')
ax.set_ylabel('Predicted Excess Heat (W)')
ax.set_title(f'Pred vs Actual (R² = {reg_result.r2:.3f})')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Сравнение 3 физических режимов

In [ ]:
from physics_engine import compare_modes, PhysicsEngine

# Сравнение при фиксированных условиях
materials = ['Pd', 'Ni', 'Fe', 'Ti', 'Au', 'Pt', 'PdO']
modes = ['maxwell', 'coulomb_original', 'cherepanov']

comparison_data = []
for mat in materials:
    results = compare_modes(material=mat, E_cm_keV=2.5, T_K=340, D_loading=0.9)
    for mode, r in results.items():
        comparison_data.append({
            'material': mat,
            'mode': mode,
            'barrier_keV': r.barrier_keV,
            'effective_barrier_keV': r.effective_barrier_keV,
            'penetration': r.penetration_probability,
            'rate': r.reaction_rate_relative,
            'log_rate': np.log10(max(r.reaction_rate_relative, 1e-300)),
        })

df_comp = pd.DataFrame(comparison_data)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Effective barrier
pivot_barrier = df_comp.pivot(index='material', columns='mode', values='effective_barrier_keV')
pivot_barrier.plot.bar(ax=axes[0], color=['#4fc3f7', '#ff7043', '#66bb6a'])
axes[0].set_title('Effective Barrier (keV)')
axes[0].set_ylabel('keV')
axes[0].legend(fontsize=8)

# Log rate
pivot_rate = df_comp.pivot(index='material', columns='mode', values='log_rate')
pivot_rate.plot.bar(ax=axes[1], color=['#4fc3f7', '#ff7043', '#66bb6a'])
axes[1].set_title('log10(Reaction Rate)')
axes[1].set_ylabel('log10(rate)')
axes[1].legend(fontsize=8)

# Penetration probability
pivot_pen = df_comp.pivot(index='material', columns='mode', values='penetration')
pivot_pen.plot.bar(ax=axes[2], color=['#4fc3f7', '#ff7043', '#66bb6a'])
axes[2].set_title('Penetration Probability')
axes[2].set_ylabel('P')
axes[2].set_yscale('log')
axes[2].legend(fontsize=8)

plt.suptitle('Сравнение 3 режимов: E=2.5 keV, T=340K, D/Pd=0.9', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Energy sweep: rate vs energy для Pd в 3 режимах
energies = np.linspace(0.5, 30, 100)

fig, ax = plt.subplots(figsize=(10, 6))
colors = {'maxwell': '#4fc3f7', 'coulomb_original': '#ff7043', 'cherepanov': '#66bb6a'}

for mode in modes:
    engine = PhysicsEngine(mode)
    rates = []
    for E in energies:
        r = engine.calculate_barrier('Pd', E, T_K=340, D_loading=0.9)
        rates.append(np.log10(max(r.reaction_rate_relative, 1e-300)))
    ax.plot(energies, rates, label=mode, color=colors[mode], linewidth=2)

ax.set_xlabel('E_cm (keV)')
ax.set_ylabel('log10(Reaction Rate)')
ax.set_title('Pd: Rate vs Energy (D/Pd=0.9, T=340K)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Numba Benchmark

In [ ]:
import time

try:
    from numba_kernels import (
        batch_gamow_penetration, batch_cross_section_DD,
        batch_barrier_maxwell, HAS_NUMBA,
    )
    from lenr_constants import gamow_penetration, cross_section_DD

    N = 100_000
    E_array = np.random.uniform(0.1, 50.0, N).astype(np.float64)
    Us_array = np.random.uniform(50, 600, N).astype(np.float64)
    D_array = np.random.uniform(0.0, 1.0, N).astype(np.float64)

    # Warm up Numba JIT
    _ = batch_gamow_penetration(E_array[:10])

    # Python loop
    t0 = time.perf_counter()
    python_result = np.array([gamow_penetration(e) for e in E_array])
    t_python = time.perf_counter() - t0

    # Numba batch
    t0 = time.perf_counter()
    numba_result = batch_gamow_penetration(E_array)
    t_numba = time.perf_counter() - t0

    speedup = t_python / t_numba
    print(f'Numba available: {HAS_NUMBA}')
    print(f'N = {N:,}')
    print(f'Python loop: {t_python:.3f}s')
    print(f'Numba batch: {t_numba:.3f}s')
    print(f'Speedup: {speedup:.1f}x')
    print(f'Max difference: {np.max(np.abs(python_result - numba_result)):.2e}')

    # Full barrier benchmark
    t0 = time.perf_counter()
    eb, pen, rate = batch_barrier_maxwell(E_array, Us_array, D_array)
    t_barrier = time.perf_counter() - t0
    print(f'\nBatch barrier Maxwell ({N:,} samples): {t_barrier:.3f}s')

except ImportError as e:
    print(f'Numba не установлен: {e}')
    print('pip install numba')

## 8. Сохранение моделей

In [ ]:
import os

save_dir = ROOT / 'saved_models'
os.makedirs(save_dir, exist_ok=True)

# Сохраняем классификатор
classifier.save(str(save_dir / 'xgboost_classifier.joblib'))
print(f'Classifier saved to {save_dir / "xgboost_classifier.joblib"}')

# Сохраняем регрессор
regressor.save(str(save_dir / 'dnn_regressor.pt'))
print(f'Regressor saved to {save_dir / "dnn_regressor.pt"}')

# Сохраняем детектор аномалий
detector.save(str(save_dir / 'anomaly_detector.joblib'))
print(f'Anomaly detector saved to {save_dir / "anomaly_detector.joblib"}')

# Сохраняем данные
data_dir = ROOT / 'saved_data'
os.makedirs(data_dir, exist_ok=True)
df_synthetic.to_csv(str(data_dir / 'synthetic_data.csv'), index=False)
df_clean.to_csv(str(data_dir / 'clean_data.csv'), index=False)
print(f'\nData saved to {data_dir}')

In [ ]:
# Загрузка моделей (проверка)
clf_loaded = LENRClassifier.load(str(save_dir / 'xgboost_classifier.joblib'))
reg_loaded = LENRRegressor.load(str(save_dir / 'dnn_regressor.pt'))

# Предсказание на тестовой выборке
test_sample = df_clean.sample(5, random_state=123)
print('Classifier predictions:')
probs = clf_loaded.predict(test_sample)
for i, (_, row) in enumerate(test_sample.iterrows()):
    print(f"  {row['material']} | E={row['beam_energy_keV']:.1f}keV | D/Pd={row['deuterium_loading']:.2f} | P(reaction)={probs[i]:.3f}")

print('\nRegressor predictions:')
heats = reg_loaded.predict(test_sample)
for i, (_, row) in enumerate(test_sample.iterrows()):
    print(f"  {row['material']} | actual={row['excess_heat_W']:.2f}W | predicted={heats[i]:.2f}W")

## 9. Итоговая сводка

In [ ]:
print('=' * 60)
print('LENR Alternative Physics — Training Summary')
print('=' * 60)
print(f'\nДанные:')
print(f'  Синтетических: {len(df_synthetic)}')
print(f'  После фильтрации: {len(df_clean)}')
print(f'  Экспериментальных: {len(df_experimental)}')
print(f'  Features: {len(feature_cols)}')
print(f'\nXGBoost Classifier:')
print(f'  AUC-ROC: {clf_result.auc_roc:.4f}')
print(f'  F1: {clf_result.f1:.4f}')
if clf_result.cv_scores is not None:
    print(f'  CV AUC: {clf_result.cv_scores.mean():.4f} ± {clf_result.cv_scores.std():.4f}')
print(f'  Top features: {", ".join(list(clf_result.feature_importance.keys())[:5])}')
print(f'\nDNN Regressor:')
print(f'  R²: {reg_result.r2:.4f}')
print(f'  RMSE: {reg_result.rmse:.4f}')
print(f'  Epochs: {len(reg_result.train_losses)}')
print(f'  Top features: {", ".join(list(reg_result.feature_importance.keys())[:5])}')
print(f'\nAnomalies: {anomaly_result.n_anomalies} ({anomaly_result.anomaly_fraction:.1%})')
print(f'\nФизические режимы:')
print(f'  Maxwell — стандартный кулоновский барьер + экранирование')
print(f'  Coulomb Original — масса электричества, плотностное взаимодействие')
print(f'  Cherepanov — фотонная масса, нет заряда, магнитный поток')